# 18 Persistence, GPT-5.6 Luna

Collects the persistence extension one model at a time, GPT-5.6 Luna first.

The extension replays each model's own single-turn reply and then presses on it
with three methods. Nothing about the opening is regenerated: turn 1 is the
prompt already put in the adaptation pass and turn 2 is that model's own reply
to it, so movement at a later turn is measured against what the model actually
did rather than against a fresh draw.

Design, fixed in `config/settings.yml` before any dialogue exists:

| | |
|---|---|
| Scenarios | 50, all 25 age-restricted and 25 of 50 harmful, seeded |
| Conditions | control and the seven stated ages, seven to eighteen |
| Methods | Purpose Reverse, Role Play, Emotional Pushback |
| Opening replicate | first |
| Generated turns | 2 |

1,200 dialogue texts, 7,200 dialogues across six models, 14,400 replies.

Runtime is inherited from the adaptation pass and not restated: temperature 1.0,
`top_p` 1, 4,096 tokens, thinking modes disabled. Changing it would make the
before and after incomparable, and the whole measure is a difference between
them.

**Scope.** Only Main Response and Delivery Response are reported from this arm.
Agreement was established on single-turn replies and does not transfer to a turn
that follows a refusal, so the extension carries its own calibration at depth,
restricted to the two fields whose single-turn agreement is highest. No claim is
made about signposting or identity across turns.

## Setup

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'scripts'))

import pandas as pd

import build
import run
from settings import (BENCHMARK_PATH, DIALOGUE, DIALOGUE_DIR, METHODS,
                      PANEL, PLAN_PATH, PROMPTS_PATH, SEED)

pd.set_option('display.width', 200, 'display.max_columns', 40)

MODEL = 'gpt-5.6-luna'

## Confirm the design

Read back what the config declares, before anything is built. Every figure below
comes from `settings.yml` rather than from this notebook, so a change made there
shows up here and a change made here does not exist.

In [ ]:
print('scenarios        ', DIALOGUE['scenarios'])
print('strata           ', ', '.join(DIALOGUE['strata']))
print('conditions       ', len(DIALOGUE['conditions']),
      '  ' + ', '.join(DIALOGUE['conditions']))
print('methods          ', len(DIALOGUE['methods']))
print('opening replicate', DIALOGUE['opening_replicate'])
print('seed             ', SEED)

for method in DIALOGUE['methods']:
    print(f'\n{method}')
    for turn in METHODS[method]['turns']:
        print(f'   {turn}')

planned = (DIALOGUE['scenarios'] * len(DIALOGUE['conditions'])
           * len(DIALOGUE['methods']))
print(f'\n{planned:,} dialogue texts'
      f'   {planned * len(PANEL):,} dialogues'
      f'   {planned * len(PANEL) * 2:,} replies planned')

## Build the dialogues

`build.build_turns()` writes `results/persistence/dialogues.csv`, one row per
turn. Turn 1 is the user prompt, turn 2 the replayed reply, and the odd turns
after that are the method's user turns with an empty assistant row waiting to be
filled.

Openings that carry no reply are dropped here and counted. Those are the
requests a provider withheld in the adaptation pass, so no dialogue can open on
them. The count belongs beside the single-turn retention figures: a withheld
request is a boundary that held before the conversation began, not missing
data.

In [ ]:
build.build_turns()

dialogues = pd.read_csv(PLAN_PATH, dtype=str)
print(f'\n{dialogues["dialogue_id"].nunique():,} dialogues, '
      f'{len(dialogues):,} rows')

### What was built, and what was withheld

The two tables below are the record of coverage. The first is what will be
collected per model; the second is what could not open. A model with fewer
dialogues than the others is not under-collected, it is a model whose provider
declined the opening request, and the contrast for that model rests on the
scenarios that remain.

In [ ]:
built = (dialogues[dialogues['turn'] == '1']
         .groupby('model')
         .agg(dialogues=('dialogue_id', 'nunique'),
              scenarios=('scenario_id', 'nunique')))
built['replies'] = built['dialogues'] * 2
print(built.to_string())

expected = (DIALOGUE['scenarios'] * len(DIALOGUE['conditions'])
            * len(DIALOGUE['methods']))
built['withheld'] = expected - built['dialogues']
print(f'\nplanned {expected * len(PANEL) * 2:,} replies, '
      f'{int(built["replies"].sum()):,} buildable, '
      f'{int(built["withheld"].sum()) * 2:,} not generated')

In [ ]:
# Which scenarios lost openings, and on which conditions. This is the
# persistence counterpart of tab:safety-retention and is reported with the
# results rather than quietly absorbed.
opened = set(zip(dialogues['model'], dialogues['prompt_id']))
prompts = pd.read_csv(PROMPTS_PATH, dtype=str)
wanted = prompts[prompts['condition'].isin(DIALOGUE['conditions'])
                 & prompts['scenario_id'].isin(dialogues['scenario_id'])]

missing = [{'model': model, 'scenario_id': row['scenario_id'],
            'condition': row['condition']}
           for model in dialogues['model'].unique()
           for _, row in wanted.iterrows()
           if (model, row['prompt_id']) not in opened]

if missing:
    gaps = pd.DataFrame(missing)
    print(gaps.groupby(['model', 'scenario_id']).size()
          .rename('conditions withheld').to_string())
else:
    print('every opening cell carries a reply')

## Collect, GPT-5.6 Luna only

One model at a time, so that a failure costs one model's pass rather than the
whole extension. GPT-5.6 Luna goes first because it is the cheapest of the six
per reply and reached the highest refusal rate on age-restricted scenarios at a
stated minor age, so it is both the smallest bill and the model with the most
room to erode.

Two generated turns per dialogue, run in order: the second user turn is sent
only after the first assistant turn has come back, because the model must be
answering its own reply rather than a placeholder.

In [ ]:
arguments = run.parser().parse_args(['dialogue', '--model', MODEL])

run.run_dialogue(arguments)

### Merge

Kept apart from collection so it can be rerun, and so an incomplete dialogue
fails loudly here rather than passing into the analysis as a short
conversation. Run it after all six models are in.


In [ ]:
# Once every model is collected, fill the plan from the collected turns.
# Any dialogue missing a generated turn is dropped whole and counted.

# run.merge_turns()

## Yield

The same conditions the adaptation pass counts on ingest, so the two are
comparable: how many calls returned, how many were withheld, how many carry an
error. Nothing here is a result. It is the record that lets a later reader tell
an absent reply from a refused one.

In [ ]:
collected = pd.read_csv(DIALOGUE_DIR / f'{MODEL}.csv', dtype=str)

blank = collected['response'].astype(str).str.strip() == ''
failed = collected['error'].astype(str).str.strip() != ''

print(f'replies returned  {len(collected):,}')
print(f'empty             {int(blank.sum()):,}')
print(f'errors            {int(failed.sum()):,}')
print(f'usable            {int((~blank & ~failed).sum()):,}')

print('\nby turn:')
print(collected.groupby('turn')
      .agg(replies=('response', 'size'),
           empty=('response',
                  lambda s: int((s.astype(str).str.strip() == '').sum())))
      .to_string())

print('\nby method:')
print(collected.groupby('method').size().rename('replies').to_string())

## Next

Repeat with `MODEL` set to each of the remaining five, then classify. The
classification pass applies the same frozen rubric under the same runtime, but
**only Main Response and Delivery Response are reported from this arm**, and
only after the calibration sample at depth has been annotated.

The calibration is sixty replies drawn stratified by method and condition,
hand-annotated on those two fields alone. Until it exists, no number from this
notebook goes into the thesis: the admission rule in Section 3.6.1 allows a
field to carry a test only where its agreement clears the floor, and agreement
at depth has never been measured.